# Momentum strategy comparison

Compare configurable momentum allocation variants against the baseline strategies from the validation notebook. All strategies are evaluated on a common date window aligned to the longest momentum warmup period.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from asset_allocation.backtest import run_backtest
from asset_allocation.config import BacktestConfig
from asset_allocation.data import load_prices
from asset_allocation.metrics import summary
from asset_allocation.plotting import (
    plot_allocation,
    plot_drawdown,
    plot_equity_curve,
    plot_summary,
)
from asset_allocation.strategy import (
    buy_and_hold_weights,
    constant_weights,
    momentum_weights,
    periodic_weights,
)

%matplotlib inline

## Load prices

In [ ]:
prices_full = load_prices()
print(
    f"Full sample: {len(prices_full)} months | "
    f"{prices_full.index[0].date()} -> {prices_full.index[-1].date()}"
)

## Strategy definitions

**Baselines** (from validation notebook):
- 100% momentum / 100% value
- 50/50 annual rebalance
- 50/50 buy-and-hold

**Momentum variants** vary lookback windows, volatility scaling, winner vs relative allocation, momentum-gap thresholds, and optional absolute (cash) mode.

In [ ]:
BASELINE_STRATEGIES = {
    "100% momentum": lambda p: constant_weights(p, {"momentum": 1.0}),
    "100% value": lambda p: constant_weights(p, {"value": 1.0}),
    "50/50 annual": lambda p: periodic_weights(p, {"momentum": 0.5, "value": 0.5}, freq="Y"),
    "50/50 buy-and-hold": lambda p: buy_and_hold_weights(p, {"momentum": 0.5, "value": 0.5}),
}

MOMENTUM_CONFIGS: dict[str, dict] = {
    "mom 12m winner": {"lookbacks": (12,), "allocation": "winner"},
    "mom 12m relative": {"lookbacks": (12,), "allocation": "relative"},
    "mom 6m winner": {"lookbacks": (6,), "allocation": "winner"},
    "mom 3m winner": {"lookbacks": (3,), "allocation": "winner"},
    "mom 6+12 blend winner": {"lookbacks": (6, 12), "allocation": "winner"},
    "mom 6+12 blend relative": {"lookbacks": (6, 12), "allocation": "relative"},
    "mom 12m vol-scaled winner": {
        "lookbacks": (12,),
        "vol_scaled": True,
        "allocation": "winner",
    },
    "mom 12m vol-scaled relative": {
        "lookbacks": (12,),
        "vol_scaled": True,
        "allocation": "relative",
    },
    "mom 6+12 vol-scaled winner": {
        "lookbacks": (6, 12),
        "vol_scaled": True,
        "allocation": "winner",
    },
    "mom 6+12 vol-scaled relative": {
        "lookbacks": (6, 12),
        "vol_scaled": True,
        "allocation": "relative",
    },
    "mom 12m winner gap 2%": {
        "lookbacks": (12,),
        "allocation": "winner",
        "threshold": 0.02,
    },
    "mom 12m vol-scaled gap 2%": {
        "lookbacks": (12,),
        "vol_scaled": True,
        "allocation": "winner",
        "threshold": 0.02,
    },
    "mom 12m absolute (cash)": {
        "lookbacks": (12,),
        "allocation": "winner",
        "absolute": True,
    },
}


def first_rebalance_date(prices: pd.DataFrame, **kwargs) -> pd.Timestamp:
    weights = momentum_weights(prices, **kwargs)
    defined = weights.dropna(how="all")
    if defined.empty:
        raise ValueError(f"No rebalance dates for config: {kwargs}")
    return defined.index[0]


common_start = max(
    first_rebalance_date(prices_full, **cfg) for cfg in MOMENTUM_CONFIGS.values()
)
prices = prices_full.loc[common_start:].copy()
print(f"Common comparison window: {prices.index[0].date()} -> {prices.index[-1].date()} ({len(prices)} months)")

In [ ]:
strategies: dict[str, pd.DataFrame] = {}

for name, builder in BASELINE_STRATEGIES.items():
    strategies[name] = builder(prices)

for name, cfg in MOMENTUM_CONFIGS.items():
    strategies[name] = momentum_weights(prices, **cfg)

config = BacktestConfig()

## Run backtests (German taxes & costs)

In [ ]:
results = {
    name: run_backtest(prices, weights, config)
    for name, weights in strategies.items()
}

summary_df = pd.concat([summary(r, name) for name, r in results.items()])
summary_df = summary_df.sort_values("sharpe_ratio", ascending=False)

summary_df.style.format(
    {
        "total_return": "{:.1%}",
        "cagr": "{:.1%}",
        "annual_volatility": "{:.1%}",
        "sharpe_ratio": "{:.2f}",
        "sortino_ratio": "{:.2f}",
        "calmar_ratio": "{:.2f}",
        "max_drawdown": "{:.1%}",
        "best_month": "{:.1%}",
        "worst_month": "{:.1%}",
        "win_rate": "{:.1%}",
        "annual_turnover": "{:.2f}",
        "time_in_market": "{:.1%}",
        "total_taxes": "{:,.0f}",
        "total_costs": "{:,.0f}",
    },
    na_rep="-",
)

## Baselines vs top momentum variants

In [ ]:
top_momentum = summary_df.drop(BASELINE_STRATEGIES.keys()).head(3).index.tolist()
focus_names = list(BASELINE_STRATEGIES.keys()) + top_momentum
focus_results = {name: results[name] for name in focus_names}

plot_summary(focus_results)
plt.show()

## Momentum hyperparameter heatmap

CAGR and Sharpe for a subset of lookback × allocation × vol-scaling combinations.

In [ ]:
HEATMAP_CONFIGS = {
    (3, False, "winner"): {"lookbacks": (3,), "allocation": "winner"},
    (6, False, "winner"): {"lookbacks": (6,), "allocation": "winner"},
    (12, False, "winner"): {"lookbacks": (12,), "allocation": "winner"},
    ((6, 12), False, "winner"): {"lookbacks": (6, 12), "allocation": "winner"},
    (3, False, "relative"): {"lookbacks": (3,), "allocation": "relative"},
    (6, False, "relative"): {"lookbacks": (6,), "allocation": "relative"},
    (12, False, "relative"): {"lookbacks": (12,), "allocation": "relative"},
    ((6, 12), False, "relative"): {"lookbacks": (6, 12), "allocation": "relative"},
    (3, True, "winner"): {"lookbacks": (3,), "vol_scaled": True, "allocation": "winner"},
    (6, True, "winner"): {"lookbacks": (6,), "vol_scaled": True, "allocation": "winner"},
    (12, True, "winner"): {"lookbacks": (12,), "vol_scaled": True, "allocation": "winner"},
    ((6, 12), True, "winner"): {"lookbacks": (6, 12), "vol_scaled": True, "allocation": "winner"},
    (3, True, "relative"): {"lookbacks": (3,), "vol_scaled": True, "allocation": "relative"},
    (6, True, "relative"): {"lookbacks": (6,), "vol_scaled": True, "allocation": "relative"},
    (12, True, "relative"): {"lookbacks": (12,), "vol_scaled": True, "allocation": "relative"},
    ((6, 12), True, "relative"): {"lookbacks": (6, 12), "vol_scaled": True, "allocation": "relative"},
}


def _lookback_label(lb: int | tuple[int, ...]) -> str:
    if isinstance(lb, tuple):
        return "+".join(str(x) for x in lb)
    return str(lb)


heatmap_rows: list[dict] = []
for (lookback, vol_scaled, allocation), cfg in HEATMAP_CONFIGS.items():
    weights = momentum_weights(prices, **cfg)
    result = run_backtest(prices, weights, config)
    metrics = summary(result).iloc[0]
    heatmap_rows.append(
        {
            "lookback": _lookback_label(lookback),
            "variant": f"{'vol' if vol_scaled else 'raw'}-{allocation}",
            "cagr": metrics["cagr"],
            "sharpe_ratio": metrics["sharpe_ratio"],
        }
    )

heatmap_df = pd.DataFrame(heatmap_rows)
lookback_order = ["3", "6", "12", "6+12"]
variant_order = ["raw-winner", "raw-relative", "vol-winner", "vol-relative"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, title in zip(
    axes,
    ["cagr", "sharpe_ratio"],
    ["CAGR", "Sharpe ratio"],
    strict=True,
):
    pivot = heatmap_df.pivot(index="lookback", columns="variant", values=metric)
    pivot = pivot.reindex(index=lookback_order, columns=variant_order)
    im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title(title)
    for i, row in enumerate(pivot.values):
        for j, val in enumerate(row):
            if np.isnan(val):
                continue
            fmt = f"{val:.1%}" if metric == "cagr" else f"{val:.2f}"
            ax.text(j, i, fmt, ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle("Momentum hyperparameter grid", y=1.02)
fig.tight_layout()
plt.show()

## Allocation over time (default 12m winner)

In [ ]:
plot_allocation(results["mom 12m winner"], title="12-month momentum winner — held weights")
plt.show()

plot_equity_curve(
    {
        "100% momentum": results["100% momentum"],
        "mom 12m winner": results["mom 12m winner"],
        "mom 12m vol-scaled winner": results["mom 12m vol-scaled winner"],
        "mom 12m absolute (cash)": results["mom 12m absolute (cash)"],
    },
    title="Momentum variants vs buy-and-hold momentum index",
)
plt.show()

plot_drawdown(
    {
        "100% momentum": results["100% momentum"],
        "mom 12m winner": results["mom 12m winner"],
        "mom 12m absolute (cash)": results["mom 12m absolute (cash)"],
    },
    title="Drawdown: momentum index vs tactical momentum",
)
plt.show()